# Adaptive Mini-GPT — Google Colab T4 Launcher (M2 Qualification)

This notebook serves as the minimal bootstrap launcher for training **Adaptive Mini-GPT** on a Google Colab T4 GPU instance.

### Design Principles
1. **Single Source of Truth**: All modeling, optimization, and evaluation logic reside in the `adaptive_mini_gpt` Python package and `amg` CLI.
2. **Two-Tier Storage**: High-speed training writes to local disk (`/content/runs/colab`), while verified checkpoints are safely published to Google Drive (`/content/drive/MyDrive/...`).
3. **Resilience**: Training survives runtime disconnections and automatically restores from the latest verified checkpoint without data loss.

## 1. Verify GPU Hardware
Ensure that Google Colab has assigned a T4 (or compatible CUDA) GPU.

In [ ]:
!nvidia-smi

## 2. Mount Google Drive
Mount Google Drive to persist checkpoints outside the ephemeral Colab VM.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 3. Clone Repository & Install Dependencies with uv

In [ ]:
import os

# Set repository directory
REPO_DIR = '/content/adaptive-mini-gpt'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hieulc/adaptive-mini-gpt.git {REPO_DIR}

%cd {REPO_DIR}
!pip install --upgrade uv
!uv sync --extra dev

## 4. Run Automated Sanity Tests on T4 Environment
Verify environment integrity before training.

In [ ]:
!uv run pytest tests/

## 5. Launch Training (Local Scratch + Google Drive Backup)
Trains with mixed-precision (`--precision fp16` / autocast + GradScaler) on CUDA.
Active checkpoints are written locally to `/content/runs/colab`, and verified backups are atomically mirrored to Google Drive.

In [ ]:
!uv run amg train \
    --config configs/tiny.yaml \
    --device cuda \
    --precision fp16 \
    --save-dir /content/runs/colab \
    --backup-dir /content/drive/MyDrive/adaptive-mini-gpt-runs/exp-001 \
    --max-steps 1000

## 6. Resume Training Across Runtime Restart
If the Colab runtime is terminated, rerun Steps 1-3, then execute this cell to restore and continue seamlessly.

In [ ]:
!uv run amg train \
    --config configs/tiny.yaml \
    --device cuda \
    --precision fp16 \
    --save-dir /content/runs/colab \
    --backup-dir /content/drive/MyDrive/adaptive-mini-gpt-runs/exp-001 \
    --resume /content/drive/MyDrive/adaptive-mini-gpt-runs/exp-001/latest.pt \
    --max-steps 2000

## 7. Evaluate and Generate Samples

In [ ]:
!uv run amg evaluate \
    --checkpoint /content/drive/MyDrive/adaptive-mini-gpt-runs/exp-001/best.pt \
    --device cuda

!uv run amg generate \
    --checkpoint /content/drive/MyDrive/adaptive-mini-gpt-runs/exp-001/best.pt \
    --prompt "From fairest " \
    --max-tokens 100 \
    --device cuda